# AGQO V3.2 research runs on Modal

This notebook is the entry point for every quantum simulation, benchmark rerun and live LLM experiment of
V3.2. It submits work to Modal and keeps a local copy of each run directory under `results/runs/`.
No circuit is simulated in this kernel: the functions live in `modal_research.py` (Modal CPU) and
`modal_app.py` (Qwen3.8-27B on a Modal H100).

Prerequisites: `modal token new` once (credentials stay in `~/.modal.toml`), the price snapshot at
`data/vn30/ohlcv_daily.csv`, and for LLM tasks the deployed `agqo-llm` app. Vietnamese guide:
`docs/vi/NGHIEN_CUU_MODAL.md`.

Set `TASKS` in the next cell (papermill parameter) to choose what runs.

In [1]:
# Parameters (papermill)
TASKS = ["smoke"]          # any of: smoke, reproduce, pytest, presets, account_benchmark, llm_mandates,
                           #         llm_circuits, knn_table, knn_study
PYTEST_ARGS = []
LLM_MODEL_APP = "agqo-llm"
RUN_TAG = ""

In [2]:
# Parameters
TASKS = "[\"smoke\",\"reproduce\"]"


In [3]:
import json, os, sys, time
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "modal_research.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import modal
import modal_research as mr
from agqo.research.orchestrate import keep, revision, show

REV = revision()
print("revision", REV)
kept = {}

revision {'sha': '11e4c20f37c9b1b50c8593c7f08c1ba60681c5ee', 'short': '11e4c20', 'dirty': False}


## Smoke test: image, data snapshot and Volume

In [4]:
if "smoke" in TASKS:
    with modal.enable_output(), mr.app.run():
        res = mr.smoke.remote(REV)
    kept["smoke"] = keep(res); show(kept["smoke"])

⠋ Initializing...

✓ Initialized. View run at 
https://modal.com/apps/data1/main/ap-c5cxQ0ObNJsGnUx0ENfX4X
⠋ Initializing...

⠋ Initializing...


⠋ Creating objects...

⠸ Creating objects...

⠦ Creating objects...
├── ⠋
├── ⠋
├── ⠋
├── ⠋
├── ⠋
├── ⠋
└── ⠋

... (939 outputs left out)

⠏ Running (1/1 containers active)... View app at 
https://modal.com/apps/data1/main/ap-c5cxQ0ObNJsGnUx0ENfX4X


Stopping app - local entrypoint completed.


✓ App completed. View run at 
https://modal.com/apps/data1/main/ap-c5cxQ0ObNJsGnUx0ENfX4X


20261120-065949-11e4c20-smoke: status=ok seconds=0.7 errors=0
  data_sha256: 7f076128a4f625f9589844f7465e3a1745e1a31e2cecfa8b70e3d8e141433042


## Product failure reproduction

Runs the decision pipeline on the edge cases reported against V3.1 and audits every result on the
post-trade NAV (cash after fees + holdings). Output: `product_failures.json`.

In [5]:
if "reproduce" in TASKS:
    with modal.enable_output(), mr.app.run():
        res = mr.reproduce.remote(REV)
    kept["reproduce"] = keep(res); show(kept["reproduce"])
    pf = json.loads((kept["reproduce"] / "product_failures.json").read_text())
    for c in pf["cases"]:
        a = c.get("audit", {})
        print(f"{c['case'][:58]:58s} {c['outcome']:9s} status={c.get('status')} app_ok={c.get('app_all_ok')} audit_ok={a.get('all_ok')}")
    print("verifier accepts swapped numbers:", pf["verifier_swap"]["swapped"]["check"]["ok"])

⠋ Initializing...

✓ Initialized. View run at 
https://modal.com/apps/data1/main/ap-uHZNm7tZq55eBKXiXaDBSI
⠋ Initializing...

⠋ Initializing...


⠋ Creating objects...

⠸ Creating objects...

⠦ Creating objects...
├── ⠋
├── ⠋
├── ⠋
├── ⠋
├── ⠋
├── ⠋
└── ⠋

... (127 outputs left out)

Stopping app - local entrypoint completed.


Runner terminated.


✓ App completed. View run at 
https://modal.com/apps/data1/main/ap-uHZNm7tZq55eBKXiXaDBSI


20261120-065958-11e4c20-product-failures: status=ok seconds=21.4 errors=0
  exceptions: 4
  silent_cap_breaches: 1
  verifier_accepts_swap: True
  clarification_blocks_trades: False
reported: 200M, banking <= 25%                             decision  status=ok app_ok=True audit_ok=False
reported: hold at most one stock (20M)                     exception status=None app_ok=None audit_ok=None
hold at most one stock (100M)                              exception status=None app_ok=None audit_ok=None
hold at most two stocks (100M)                             decision  status=ok app_ok=True audit_ok=True
20M, banking <= 25%                                        decision  status=ok app_ok=True audit_ok=True
500M, banking <= 25%                                       decision  status=ok app_ok=True audit_ok=True
100M, each stock <= 10%                                    decision  status=ok app_ok=True audit_ok=True
20M, each stock <= 2% (below one lot of every stock)       exception status=No

## Test suite (including the circuit-simulating tests) on Modal CPU

In [6]:
if "pytest" in TASKS:
    with modal.enable_output(), mr.app.run():
        res = mr.pytest_suite.remote(REV, PYTEST_ARGS)
    kept["pytest"] = keep(res); show(kept["pytest"])
    print((kept["pytest"] / "pytest.txt").read_text()[-3000:])

## Summary of this notebook execution

In [7]:
print(json.dumps({k: str(v) for k, v in kept.items()}, indent=1))

{
 "smoke": "/Users/jant/Documents/ChatGPT/AI-Quantum Challenge 2026/.claude/worktrees/agqo-portfolio-allocation-d30055/results/runs/20261120-065949-11e4c20-smoke",
 "reproduce": "/Users/jant/Documents/ChatGPT/AI-Quantum Challenge 2026/.claude/worktrees/agqo-portfolio-allocation-d30055/results/runs/20261120-065958-11e4c20-product-failures"
}
